# Lab 9 — TextCNN for Drug Reviews
Streamlined Colab edition. Full commented version is in Google Drive. Educational use only.

In [ ]:
!pip -q install ucimlrepo
import re,collections,numpy as np,torch,torch.nn as nn,torch.optim as optim
from torch.utils.data import Dataset,DataLoader
from ucimlrepo import fetch_ucirepo
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score,f1_score
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
d=fetch_ucirepo(id=461); df=d.data.features.copy(); print(df.columns)
text_col=next(c for c in df.columns if 'review' in c.lower() or 'benefit' in c.lower()); rating_col=next(c for c in df.columns if 'rating' in c.lower())
df=df[[text_col,rating_col]].dropna(); df['text']=df[text_col].astype(str); df['label']=(df[rating_col].astype(float)>=7).astype(int)
train_df,test_df=train_test_split(df,test_size=.2,random_state=42,stratify=df.label)

In [ ]:
def tok(s): return re.findall(r'[a-z]+',s.lower())
cnt=collections.Counter(w for s in train_df.text for w in tok(s)); vocab={'<PAD>':0,'<UNK>':1}; vocab.update({w:i+2 for i,(w,n) in enumerate(cnt.most_common(8000))})
def enc(s,L=120):
 a=[vocab.get(w,1) for w in tok(s)[:L]]; return a+[0]*(L-len(a))
class DS(Dataset):
 def __init__(self,df): self.x=torch.tensor([enc(s) for s in df.text],dtype=torch.long); self.y=torch.tensor(df.label.values,dtype=torch.long)
 def __len__(self): return len(self.y)
 def __getitem__(self,i): return self.x[i],self.y[i]
tr=DataLoader(DS(train_df),batch_size=64,shuffle=True); te=DataLoader(DS(test_df),batch_size=128)

In [ ]:
class TextCNN(nn.Module):
 def __init__(self,V,E=64,F=64):
  super().__init__(); self.emb=nn.Embedding(V,E,padding_idx=0); self.convs=nn.ModuleList([nn.Conv1d(E,F,k) for k in (3,4,5)]); self.fc=nn.Linear(F*3,2)
 def forward(self,x):
  x=self.emb(x).transpose(1,2); x=torch.cat([torch.relu(c(x)).amax(2) for c in self.convs],1); return self.fc(x)
model=TextCNN(len(vocab)).to(device); opt=optim.Adam(model.parameters(),lr=1e-3); loss_fn=nn.CrossEntropyLoss()

In [ ]:
for e in range(3):
 model.train()
 for x,y in tr:
  x=x.to(device); y=y.to(device); opt.zero_grad(); z=model(x); loss=loss_fn(z,y); loss.backward(); opt.step()
 print('epoch',e+1,'done')
model.eval(); yt=[]; yp=[]
with torch.no_grad():
 for x,y in te:
  pred=model(x.to(device)).argmax(1).cpu().numpy(); yt.extend(y.numpy()); yp.extend(pred)
print('Accuracy',accuracy_score(yt,yp)); print('F1',f1_score(yt,yp))